# 02 — Exploratory Data Analysis

> *"You should explore the data to gain insights... visualise the data, look for correlations, and check what kind of data you have."*  
> — Aurélien Géron, *Hands-On Machine Learning with Scikit-Learn and PyTorch* (2025), Chapter 2

## 1. Setup

We insert the `src/` directory into `sys.path` so that `client_churn_prediction` is importable without installing the package.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"Project root: {PROJECT_ROOT}")

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# Plot style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams["figure.dpi"] = 110
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

CHURN_PALETTE = {0: "#4C8CBF", 1: "#E84545"}
CHURN_LABELS  = {0: "Retained", 1: "Churned"}

print("Libraries loaded.")

## 2. Load Data

In [ ]:
from client_churn_prediction.data import load_training_frame
from client_churn_prediction.config import load_config

config = load_config(PROJECT_ROOT / "configs" / "project.toml")
df = load_training_frame(config)

TARGET = "exited"  # snake_case normalised form of 'Exited'

print(f"Loaded dataset: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

## 3. Data Profiling

Following Géron's checklist: inspect shape, data types, missing values, and basic statistics before doing anything else.

In [ ]:
print("=" * 50)
print(f"Shape       : {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
print()
print("Data types:")
print(df.dtypes.value_counts().to_string())
print()
print("Missing values:")
missing = df.isna().sum()
print(missing[missing > 0].to_string() if missing.sum() > 0 else "  None — dataset is complete.")

In [ ]:
df.describe(include="all").T.round(3)

In [ ]:
# Missing-values heatmap
# Even if there are no missing values, this cell demonstrates the pattern
# for production data that may have gaps.

fig, ax = plt.subplots(figsize=(12, 3))

missing_matrix = df.isna().astype(int)
if missing_matrix.values.sum() == 0:
    ax.text(
        0.5, 0.5, "No missing values detected in this dataset.",
        transform=ax.transAxes, ha="center", va="center", fontsize=13,
        color="green", fontweight="bold",
    )
    ax.set_axis_off()
else:
    sns.heatmap(
        missing_matrix.T, cmap="Reds", cbar=False,
        yticklabels=True, xticklabels=False, ax=ax,
    )
    ax.set_title("Missing Values Heatmap (red = missing)", pad=10)
    ax.set_xlabel("Rows")

plt.tight_layout()
plt.show()

## 4. Target Distribution

The class balance directly impacts our choice of training strategy and evaluation metrics.

In [ ]:
counts = df[TARGET].value_counts().sort_index()
pct    = df[TARGET].value_counts(normalize=True).sort_index() * 100

fig, ax = plt.subplots(figsize=(5.5, 4))

x_labels = ["Retained (0)", "Churned (1)"]
colors    = [CHURN_PALETTE[0], CHURN_PALETTE[1]]

bars = ax.bar(x_labels, counts.values, color=colors, edgecolor="white", width=0.45)

for bar, p in zip(bars, pct.values):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 60,
        f"{p:.1f}%",
        ha="center", va="bottom", fontsize=12, fontweight="bold",
    )

ax.set_title("Target Distribution: Exited", fontsize=13, pad=12)
ax.set_ylabel("Number of Customers")
ax.set_ylim(0, counts.max() * 1.2)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
plt.tight_layout()
plt.show()

print(f"\nChurn rate: {pct[1]:.1f}%  |  Imbalance ratio: {counts[0]/counts[1]:.1f}:1")

## 5. Numerical Feature Distributions

Géron advises plotting histograms of all numerical attributes to understand the scale, skewness, and presence of clipping (artificial capping) early on.

In [ ]:
numerical_features = ["age", "balance", "credit_score", "estimated_salary"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()

for ax, col in zip(axes, numerical_features):
    for label, color in CHURN_PALETTE.items():
        subset = df[df[TARGET] == label][col].dropna()
        ax.hist(
            subset, bins=35, alpha=0.55, color=color,
            label=CHURN_LABELS[label], density=True,
        )
    ax.set_title(col.replace("_", " ").title(), fontsize=11)
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

fig.suptitle("Numerical Feature Distributions by Churn Status", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

**Early observations:**
- **Age:** Churned customers tend to be older — the distribution for churners shifts notably right.
- **Balance:** Many customers have zero balance; among churners the zero-balance spike is less pronounced, suggesting that high-balance customers also churn at elevated rates.
- **Credit score and Estimated salary:** Both look approximately uniform — weaker raw discriminators, but may interact usefully with other features.

## 6. Churn Rate by Geography

Geography can encode structural differences in product mix, regulation, and customer demographics.

In [ ]:
geo_col = "geography"

geo_churn = (
    df.groupby(geo_col)[TARGET]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate", "count": "n_customers"})
    .sort_values("churn_rate", ascending=False)
)
geo_churn["churn_rate_pct"] = geo_churn["churn_rate"] * 100

fig, ax = plt.subplots(figsize=(7, 4))

bars = ax.bar(
    geo_churn.index, geo_churn["churn_rate_pct"],
    color=sns.color_palette("Blues_d", len(geo_churn)),
    edgecolor="white", width=0.5,
)

for bar, (idx, row) in zip(bars, geo_churn.iterrows()):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.3,
        f"{row['churn_rate_pct']:.1f}%\n(n={row['n_customers']:,})",
        ha="center", va="bottom", fontsize=10,
    )

ax.axhline(df[TARGET].mean() * 100, color="red", linestyle="--", linewidth=1.2, label="Overall average")
ax.set_title("Churn Rate by Geography", fontsize=13)
ax.set_ylabel("Churn Rate (%)")
ax.set_ylim(0, geo_churn["churn_rate_pct"].max() * 1.35)
ax.legend()
plt.tight_layout()
plt.show()

print(geo_churn[["n_customers", "churn_rate_pct"]].round(2).to_string())

## 7. Churn Rate by Number of Products

This is one of the most striking patterns in the dataset. Customers with 3 or 4 products have disproportionately high churn rates — an important signal for feature engineering.

In [ ]:
prod_col = "num_of_products"

prod_churn = (
    df.groupby(prod_col)[TARGET]
    .agg(["mean", "count"])
    .rename(columns={"mean": "churn_rate", "count": "n_customers"})
)
prod_churn["churn_rate_pct"] = prod_churn["churn_rate"] * 100

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax2 = ax1.twinx()

colors_bar = [CHURN_PALETTE[1] if p >= 3 else CHURN_PALETTE[0] for p in prod_churn.index]

ax1.bar(
    prod_churn.index, prod_churn["churn_rate_pct"],
    color=colors_bar, edgecolor="white", width=0.5, alpha=0.85, label="Churn Rate",
)
ax2.plot(
    prod_churn.index, prod_churn["n_customers"],
    color="grey", marker="o", linewidth=2, label="Customer Count",
)

ax1.set_title("Churn Rate by Number of Products\n(red bars = products 3 & 4 — anomalously high churn)", fontsize=12)
ax1.set_xlabel("Number of Products")
ax1.set_ylabel("Churn Rate (%)", color=CHURN_PALETTE[1])
ax2.set_ylabel("Number of Customers", color="grey")
ax1.axhline(df[TARGET].mean() * 100, color="black", linestyle="--", linewidth=1, label="Overall avg")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()

print(prod_churn[["n_customers", "churn_rate_pct"]].round(2).to_string())

**Key insight:** Customers with 3 or 4 products churn at rates far above average. This is counterintuitive — more products should imply more engagement — but may indicate over-sold customers or dissatisfaction with service complexity. This pattern motivates the `products_active_combo` and `balance_per_product` features in notebook 03.

## 8. Age Distribution by Churn Status

Age is among the strongest individual predictors. Let us visualise the difference in age distributions between churners and retained customers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# -- Left: overlapping histograms
ax = axes[0]
for label, color in CHURN_PALETTE.items():
    subset = df[df[TARGET] == label]["age"].dropna()
    ax.hist(subset, bins=30, alpha=0.55, color=color, density=True, label=CHURN_LABELS[label])
ax.set_title("Age Distribution (Overlapping Histograms)")
ax.set_xlabel("Age")
ax.set_ylabel("Density")
ax.legend()

# -- Right: violin plot
ax = axes[1]
violin_data = [df[df[TARGET] == 0]["age"].dropna(), df[df[TARGET] == 1]["age"].dropna()]
vp = ax.violinplot(violin_data, positions=[0, 1], widths=0.6, showmedians=True)
for pc, color in zip(vp["bodies"], [CHURN_PALETTE[0], CHURN_PALETTE[1]]):
    pc.set_facecolor(color)
    pc.set_alpha(0.6)
for part_name in ["cbars", "cmins", "cmaxes", "cmedians"]:
    if part_name in vp:
        vp[part_name].set_color("black")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Retained", "Churned"])
ax.set_title("Age Distribution (Violin Plot)")
ax.set_ylabel("Age")

fig.suptitle("Age by Churn Status", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print("Age statistics by churn status:")
print(df.groupby(TARGET)["age"].describe().round(1).to_string())

## 9. Correlation Matrix

Géron recommends computing pairwise correlations between numerical attributes and the target variable as an early signal of predictive power.

In [ ]:
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()

# Exclude administrative ID columns
id_cols_norm = {"row_number", "customer_id", "row_id"}
numerical_cols = [c for c in numerical_cols if c not in id_cols_norm]

corr_matrix = df[numerical_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1,
    square=True, linewidths=0.5,
    cbar_kws={"shrink": 0.75},
    ax=ax,
)
ax.set_title("Correlation Matrix — Numerical Features", fontsize=13, pad=12)
plt.tight_layout()
plt.show()

# Correlations with target
print("\nCorrelation with target (exited):")
print(corr_matrix[TARGET].drop(TARGET).sort_values(key=abs, ascending=False).round(3).to_string())

## 10. Balance Distribution — Zero-Balance Customers

A large fraction of customers carry a zero balance. This bimodal distribution is unusual and merits special treatment — we will engineer an explicit `is_zero_balance` flag in notebook 03.

In [ ]:
zero_balance_share = (df["balance"] == 0).mean() * 100
zero_by_churn = df.groupby(TARGET).apply(lambda g: (g["balance"] == 0).mean() * 100)

print(f"Overall zero-balance customers: {zero_balance_share:.1f}%")
print("Zero-balance rate by churn status:")
for label, pct in zero_by_churn.items():
    print(f"  {CHURN_LABELS[label]}: {pct:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# All balances
ax = axes[0]
ax.hist(df["balance"], bins=50, color="#4C8CBF", edgecolor="white", alpha=0.8)
ax.set_title("Balance Distribution (All Customers)")
ax.set_xlabel("Balance")
ax.set_ylabel("Count")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))

# Non-zero balances only
ax = axes[1]
non_zero = df[df["balance"] > 0]
for label, color in CHURN_PALETTE.items():
    subset = non_zero[non_zero[TARGET] == label]["balance"]
    ax.hist(subset, bins=40, alpha=0.5, color=color, density=True, label=CHURN_LABELS[label])
ax.set_title("Balance Distribution — Non-Zero Only, by Churn")
ax.set_xlabel("Balance")
ax.set_ylabel("Density")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x/1000)}k"))
ax.legend()

plt.tight_layout()
plt.show()

## 11. Active Membership & Credit Card Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, title in zip(
    axes,
    ["is_active_member", "has_cr_card"],
    ["Churn Rate by Active Membership", "Churn Rate by Credit Card Ownership"],
):
    churn_by_col = (
        df.groupby(col)[TARGET]
        .mean()
        .mul(100)
        .reset_index()
        .rename(columns={TARGET: "churn_rate"})
    )
    x_labels = {0: "No", 1: "Yes"}
    ax.bar(
        [x_labels.get(v, str(v)) for v in churn_by_col[col]],
        churn_by_col["churn_rate"],
        color=[CHURN_PALETTE[0], CHURN_PALETTE[1]] if col == "is_active_member" else ["#9EC8E0", "#5A9FBF"],
        edgecolor="white", width=0.4,
    )
    ax.axhline(df[TARGET].mean() * 100, color="black", linestyle="--", linewidth=1, label="Overall avg")
    ax.set_title(title)
    ax.set_ylabel("Churn Rate (%)")
    ax.set_ylim(0, 35)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 12. Key Insights Summary

After this exploratory pass, the following patterns stand out and will directly inform our feature engineering strategy:

| Feature | Observation | Implication for modelling |
|---|---|---|
| **Age** | Churners are significantly older (median ~45 vs ~37) | `is_senior` flag (age ≥ 50) will be a strong engineered feature |
| **Number of products** | Products 3 & 4 have very high churn (>80%) despite small population | Non-linear relationship; tree models will capture this well |
| **Balance** | Bimodal — ~30% of customers have exactly zero balance | Engineer `is_zero_balance` and `balance_per_product` |
| **Active membership** | Inactive members churn ~2× more than active | Strong predictor; combine with age: `senior_inactive` |
| **Geography** | Germany churns at ~32%, France/Spain at ~17% | Geography is categorical — needs encoding |
| **Credit score / salary** | Weak direct correlation with churn | May contribute via interactions; `credit_score_band` |
| **Class balance** | ~20% churn rate | Use `class_weight='balanced'`; evaluate with AP and Recall@k |

These insights map directly to the engineered features defined in `src/client_churn_prediction/features.py` and explored in notebook 03.